# Causal Discovery and Causal Feature Selection for Robust Prediction

This tutorial walks through the key ideas and algorithms at the intersection of causal inference and robust machine learning.
The tutorial is structured as follows:

1. **Pairwise causal discovery** — can we tell which variable causes which, from observational data alone?
2. **Multivariate causal discovery** — recovering the full causal graph over many variables
3. **Multivariate causal feature selection** — finding the Markov Blanket of a target variable
4. **CFS for robust prediction** — why causal parents are more stable predictors than correlated features
5. **Invariant Causal Predictors** — a principled method to find invariant features across environments

In [ ]:
import os
if not os.path.exists('zh03-causal-discovery-robust-predictions'):
    !git clone https://github.com/WinterSchool2026/zh03-causal-discovery-robust-predictions.git
import sys
sys.path.insert(0, 'zh03-causal-discovery-robust-predictions')

In [ ]:
pip install causal-learn

In [ ]:
# Standard library
import time

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# Causal feature selection
from src.causal_feature_selection import (
    hiton_mb, iamb, mb_ges, resit_mb, hiton_pc
)
from src.kci import kci_test

# LinGAM (with automatic backend detection)
from src.linGAM import direct_lingam

# Data generation
from src.generate_scm import *

np.random.seed(2)


---
## Section 3 — Multivariate Causal Feature Selection

### 3.1 The Markov Blanket

Given a target variable Y, the **Markov Blanket (MB)** of Y is the minimal set of variables MB(Y) such that Y is conditionally independent of all other variables given MB(Y):

$$Y \perp \mathbf{X}_{\text{rest}} \mid MB(Y)$$

This means that the markov blanket in the information bottleneck between the covariates and the target variable.

In a DAG, MB(Y) = parents(Y) ∪ children(Y) ∪ spouses(Y), where spouses are the other parents of Y's children.

The MB has a key property for feature selection: it is the minimal sufficient set for predicting Y. No variable outside the MB adds any predictive information once we condition on MB(Y).

### 3.2 Why not just use the full graph?

Running a full causal discovery algorithm (PC, GES) and then extracting the MB is one option. But:
- This computationally very expensive
- Errors in distant parts of the graph can propagate to MB estimation
- MB-specific algorithms are more efficient

### 3.3 MB algorithms

**HITON-MB**: Constraint-based. Builds the MB incrementally using a HITON-PC phase (find parents/children) followed by a spouse discovery phase.

**IAMB**: Incremental Association MB. Two phases: grow (add variables that are associated with Y given current MB) and shrink (remove variables that are conditionally independent of Y given the rest of the MB).

**MB-GES**: Score-based. Uses GES restricted to the neighborhood of Y.

**RESIT-MB**: Functional. Extends the pairwise RESIT idea to identify the MB via regression and independence testing.


In [ ]:
# Generate a slightly denser SCM
# We use uniform noise so that LinGAM's non-Gaussianity assumption holds.
d = 10
scm = SCMGenerator(d=d)
scm.fit(n_parents=2, n_childs=2, n_spouses=2, sparsity=0.1,
        noise_type='uniform', is_linear=True)
col_names = np.hstack((np.array([f'X{i}' for i in range(scm.n_nodes-1)]),
                        np.array(['Y'])))
plot_graphs_from_adj([scm.A], [d], [col_names], ['True DAG'])

print(f'True parents of Y:  {sorted(scm.parents_idx)}')
print(f'True children of Y: {sorted(scm.children_idx)}')
print(f'True spouses of Y:  {sorted(scm.spouses_idx)}')
true_mb      = set(scm.parents_idx) | set(scm.children_idx) | set(scm.spouses_idx)
true_parents = set(scm.parents_idx)
print(f'True MB of Y:       {sorted(true_mb)}')
print(f'True parents of Y:  {sorted(true_parents)}')


In [ ]:
# Sample data
data = scm.sample(n_samples=20000)
data.head()


### 3.4 Run four MB algorithms and compare


In [ ]:
# HITON-MB
start = time.time()
mb_hiton = hiton_mb(data, 'Y', alpha=0.05, ci_method='partial')
hiton_time = time.time() - start
print(f"HITON-MB  : {hiton_time:.2f}s  ->  {sorted(mb_hiton)}")

# IAMB
start = time.time()
mb_iamb = iamb(data, 'Y', alpha=0.05, ci_method='partial')
iamb_time = time.time() - start
print(f"IAMB      : {iamb_time:.2f}s  ->  {sorted(mb_iamb)}")

# MB-GES
start = time.time()
mb_ges_result = mb_ges(data, 'Y')
ges_time = time.time() - start
print(f"MB-GES    : {ges_time:.2f}s  ->  {sorted(mb_ges_result)}")

# RESIT-MB
start = time.time()
mb_resit = resit_mb(data, 'Y', alpha=0.05)
resit_time = time.time() - start
print(f"RESIT-MB  : {resit_time:.2f}s  ->  {sorted(mb_resit)}")

print(f"\nTrue MB   :         ->  {sorted(true_mb)}")
